In [1]:
import pandas as pd
import requests

print("Environment and packages loaded successfully!")

Environment and packages loaded successfully!


In [2]:
import numpy as np

# Set a seed so our "random" data is exactly the same every time we run it
np.random.seed(42)

# Generate 100 fake flights with realistic variables
prototype_data = {
    'flight_hour': np.random.randint(6, 24, 100), # Flights between 6 AM and Midnight
    'weather': np.random.choice(['Clear', 'Rain', 'Storm', 'Snow'], 100, p=[0.6, 0.2, 0.1, 0.1]),
    'airline': np.random.choice(['Lufthansa', 'Ryanair', 'EasyJet'], 100),
    'is_delayed': np.random.choice([0, 1], 100, p=[0.7, 0.3]) # 0 = On Time, 1 = Delayed
}

# Convert the dictionary into a pandas DataFrame (a structured table)
df = pd.DataFrame(prototype_data)

# Display the first 5 rows
df.head()

,flight_hour,weather,airline,is_delayed
0,12,Snow,Ryanair,0
1,20,Storm,Lufthansa,0
2,16,Rain,Lufthansa,1
3,13,Clear,EasyJet,0
4,12,Clear,EasyJet,1


In [3]:
# Convert text columns ('weather' and 'airline') into binary numbers (0 or 1)
# dtype=int ensures the output is 0 and 1 instead of True and False
df_encoded = pd.get_dummies(df, columns=['weather', 'airline'], dtype=int)

# Look at how the table structure has changed
df_encoded.head()


,flight_hour,is_delayed,weather_Clear,weather_Rain,weather_Snow,weather_Storm,airline_EasyJet,airline_Lufthansa,airline_Ryanair
0,12,0,0,0,1,0,0,0,1
1,20,0,0,0,0,1,0,1,0
2,16,1,0,1,0,0,0,1,0
3,13,0,1,0,0,0,1,0,0
4,12,1,1,0,0,0,1,0,0


In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. GENERATE DATA
np.random.seed(42)
prototype_data = {
    'flight_hour': np.random.randint(6, 24, 100),
    'weather': np.random.choice(['Clear', 'Rain', 'Storm', 'Snow'], 100, p=[0.6, 0.2, 0.1, 0.1]),
    'airline': np.random.choice(['Lufthansa', 'Ryanair', 'EasyJet'], 100),
    'is_delayed': np.random.choice([0, 1], 100, p=[0.7, 0.3])
}
df = pd.DataFrame(prototype_data)

# 2. ENCODE (Text ➔ Math)
df_encoded = pd.get_dummies(df, columns=['weather', 'airline'], dtype=int)

# 3. SPLIT (80% Train / 20% Test)
y = df_encoded['is_delayed']
X = df_encoded.drop('is_delayed', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. TRAIN MODEL
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 5. PREDICT & GRADE
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 65.00%


In [15]:
import requests
import pandas as pd

# 1. Define the API endpoint and the geographic bounding box for Munich Airspace
url = "https://opensky-network.org/api/states/all"
munich_airspace = {
    'lamin': 48.0,  # Latitude minimum
    'lamax': 48.5,  # Latitude maximum
    'lomin': 11.4,  # Longitude minimum
    'lomax': 12.0   # Longitude maximum
}

print("Pinging OpenSky API for live Munich traffic...")

# 2. Make the request to the live server
response = requests.get(url, params=munich_airspace)

# 3. Check if the connection was successful (200 OK)
if response.status_code == 200:
    data = response.json()
    
    # 4. Extract the raw state vectors and define the columns
    flight_vectors = data['states']
    columns = ['icao24', 'callsign', 'origin_country', 'time_position', 
               'last_contact', 'longitude', 'latitude', 'baro_altitude', 
               'on_ground', 'velocity', 'true_track', 'vertical_rate', 
               'sensors', 'geo_altitude', 'squawk', 'spi', 'position_source']
    
    # 5. Convert the messy JSON feed into a clean pandas DataFrame
    live_flights_df = pd.DataFrame(flight_vectors, columns=columns)
    
    print(f"Success! Tracking {len(live_flights_df)} aircraft currently in the sector.")
    
    # Display the most readable columns
    display(live_flights_df[['callsign', 'origin_country', 'baro_altitude', 'velocity','icao24','on_ground']].head())
else:
    print(f"Failed to connect. Error code: {response.status_code}")


Pinging OpenSky API for live Munich traffic...
Success! Tracking 9 aircraft currently in the sector.


,callsign,origin_country,baro_altitude,velocity,icao24,on_ground
0,THY1UP,Turkey,NaN,0.45,4bb28b,True
1,GEC8290,Germany,10668.00,268.05,3c70cc,False
2,DLH5AH,Germany,449.58,72.10,3c65cf,False
3,DLH6EE,Germany,NaN,0.00,3c6747,True
4,DLH715,Germany,480.06,72.49,3c671a,False


In [16]:
# 1. Create our two mathematical rules based on the data dictionary
is_flying = live_flights_df['on_ground'] == False
is_descending = live_flights_df['vertical_rate'] < 0

# 2. Apply the rules to filter the table, and drop any rows missing a callsign
arriving_flights = live_flights_df[is_flying & is_descending].dropna(subset=['callsign'])

print(f"Found {len(arriving_flights)} aircraft currently on descent.")

# 3. Display the cleaned, filtered table
display(arriving_flights[['callsign', 'origin_country', 'baro_altitude', 'vertical_rate']])

Found 5 aircraft currently on descent.


,callsign,origin_country,baro_altitude,vertical_rate
2,DLH5AH,Germany,449.58,-3.90
4,DLH715,Germany,480.06,-3.25
5,DLH01A,Germany,1082.04,-5.20
6,DKBUW,Germany,495.30,-0.33
8,EDL7,Germany,533.40,-0.65


In [18]:
import numpy as np

# 1. Munich Airport (MUC / EDDM) precise GPS coordinates
MUC_LAT = 48.3538
MUC_LON = 11.7861

# 2. Define the Haversine formula (Standard geospatial math)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth's radius in kilometers
    
    # Convert degrees to radians for the math to work
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return R * c

# 3. Apply the math to our radar data (.copy() prevents pandas warnings)
arriving_flights = arriving_flights.copy()
arriving_flights['distance_to_muc_km'] = haversine_distance(
    arriving_flights['latitude'], 
    arriving_flights['longitude'], 
    MUC_LAT, 
    MUC_LON
)

# 4. Sort the table to show the closest planes first
arriving_flights = arriving_flights.sort_values(by='distance_to_muc_km')

# 5. Display the results
display(arriving_flights[['callsign', 'distance_to_muc_km', 'velocity', 'baro_altitude']].head())

,callsign,distance_to_muc_km,velocity,baro_altitude
2,DLH5AH,3.191486,72.10,449.58
4,DLH715,3.904502,72.49,480.06
5,DLH01A,14.954271,94.37,1082.04
8,EDL7,29.470909,5.75,533.40
6,DKBUW,29.930745,30.97,495.30
